# 04 - Score the fine-tuned VLM on all 863 eval images (Week 3, part 1)

For every eval image the model answers in JSON, and we also record a **score**: the model's probability that `"defective"` is `true`. AUROC needs that score; a plain yes/no is not enough.

We also score **100 of its own training images**. If it is far better on those than on eval, it memorised (overfit).

**Settings:** GPU T4 x2 · Internet On · Inputs: **MVTec AD**, **mvtec-split**, **mvtec-finetune** (for the adapters).
**Time:** roughly 30-60 minutes, so run it with **Save Version → Save & Run All** and close the tab.

## Step 1 - Install Unsloth

In [ ]:
%%capture
!pip install -q unsloth

## Step 2 - GPU, split and adapters

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"
print(torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
from pathlib import Path

split_files = list(Path('/kaggle/input').rglob('splits.csv'))
assert split_files, "splits.csv not found: Add Input -> Your Work -> mvtec-split"
df = pd.read_csv(split_files[0])
roots = [p.parent for p in Path('/kaggle/input').rglob('bottle') if p.is_dir() and (p / 'train').exists()]
assert roots, "MVTec AD not found: Add Input -> the MVTec AD dataset"
ROOT = roots[0]

def rebase(p):
    if not isinstance(p, str):
        return p
    parts = p.split('/')
    i = parts.index('ground_truth') - 1 if 'ground_truth' in parts else len(parts) - 4
    return str(ROOT.joinpath(*parts[i:]))

df['path'] = df['path'].map(rebase)
df['mask_path'] = df['mask_path'].map(rebase)
assert Path(df.path.iloc[0]).exists(), "paths don't resolve"
print(len(df), 'rows |', df.role.value_counts().to_dict())

In [ ]:
import shutil
cfgs = list(Path('/kaggle/input').rglob('adapter_config.json'))
assert cfgs, "Adapters not found: Add Input -> Your Work -> mvtec-finetune"
ADAPTER_DIR = '/kaggle/working/qwen2vl-mvtec-lora'
shutil.copytree(cfgs[0].parent, ADAPTER_DIR, dirs_exist_ok=True)   # input folders are read-only
print('adapters:', sorted(os.listdir(ADAPTER_DIR)))

## Step 3 - Helpers (same prompt and image size as training)

In [ ]:
import json, math
import numpy as np
from PIL import Image

IMG_SIZE = 448

def make_prompt(category, allowed_types):
    name = category.replace("_", " ")
    types = ", ".join(sorted(allowed_types))
    return (f"You are inspecting a {name} on a factory line. Is it defective? "
            f"Possible defect types: {types}. "
            'Answer only with JSON: {"defective": true/false, "defect_type": "<type or none>", '
            '"location": "<top-left|top|top-right|left|center|right|bottom-left|bottom|bottom-right|none>"}')

def load_image(path):
    return Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)

def parse_json(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        return json.loads(text[start:end])
    except ValueError:
        return None

GRID = [["top-left", "top", "top-right"], ["left", "center", "right"], ["bottom-left", "bottom", "bottom-right"]]
def location_from_mask(mask_path):
    m = np.array(Image.open(mask_path).convert("L")) > 0
    if not m.any():
        return "none"
    ys, xs = np.nonzero(m); h, w = m.shape
    return GRID[min(int(ys.mean() / h * 3), 2)][min(int(xs.mean() / w * 3), 2)]

## Step 4 - Load base model + your adapters

In [ ]:
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True)
FastVisionModel.for_inference(model)

tok = getattr(tokenizer, 'tokenizer', tokenizer)
T_ID = tok.encode(' true', add_special_tokens=False)[0]
F_ID = tok.encode(' false', add_special_tokens=False)[0]
assert T_ID != F_ID
allowed = df[df.label == 'defect'].groupby('category').defect_type.unique().to_dict()
print('loaded | token ids true/false:', T_ID, F_ID)

## Step 5 - The scoring function
One generation per image. The score is read at the moment the model writes `true` or `false` after `"defective":`. If that fails, a backup forward pass reads it directly.

In [ ]:
def build_inputs(row, prefix=''):
    msgs = [{"role": "user", "content": [{"type": "image"},
             {"type": "text", "text": make_prompt(row['category'], allowed[row['category']])}]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True) + prefix
    return tokenizer(load_image(row['path']), text, add_special_tokens=False, return_tensors="pt").to("cuda")

@torch.no_grad()
def backup_score(row):
    inputs = build_inputs(row, prefix='{"defective":')
    logits = model(**inputs).logits[0, -1, [T_ID, F_ID]].float()
    return torch.softmax(logits, 0)[0].item()

@torch.no_grad()
def score_one(row):
    inputs = build_inputs(row)
    out = model.generate(**inputs, max_new_tokens=48, do_sample=False, use_cache=True,
                         return_dict_in_generate=True, output_scores=True)
    gen = out.sequences[0, inputs['input_ids'].shape[1]:]
    score = float('nan')
    for step, tid in enumerate(gen.tolist()):
        if out.scores is None:
            break
        if tid in (T_ID, F_ID):
            score = torch.softmax(out.scores[step][0, [T_ID, F_ID]].float(), 0)[0].item()
            break
    if math.isnan(score):
        score = backup_score(row)
    raw = tok.decode(gen, skip_special_tokens=True).strip()
    pred = parse_json(raw) or {}
    return dict(score=score, raw=raw, valid_json=bool(pred),
                pred_defective=pred.get('defective'), pred_type=pred.get('defect_type'),
                pred_location=pred.get('location'))

r0 = df[df.role == 'eval'].iloc[0]
print(r0.category, r0.defect_type, '->', score_one(r0))

## Step 6 - Score everything (saves progress every 50 images)

In [ ]:
import transformers, time
from tqdm.auto import tqdm
transformers.logging.set_verbosity_error()   # hides the repeated max_length message

def run(rows, out_csv):
    done = pd.read_csv(out_csv) if Path(out_csv).exists() else pd.DataFrame()
    seen = set(done.path) if len(done) else set()
    recs = done.to_dict('records')
    for i, (_, r) in enumerate(tqdm(rows.iterrows(), total=len(rows), desc=Path(out_csv).stem)):
        if r.path in seen:
            continue
        rec = dict(path=r.path, category=r.category, kind=r.kind, label=r.label, defect_type=r.defect_type,
                   true_location=location_from_mask(r.mask_path) if r.label == 'defect' else 'none')
        rec.update(score_one(r))
        recs.append(rec)
        if len(recs) % 50 == 0:
            pd.DataFrame(recs).to_csv(out_csv, index=False)
    out = pd.DataFrame(recs); out.to_csv(out_csv, index=False)
    return out

t0 = time.time()
ev = run(df[df.role == 'eval'], '/kaggle/working/vlm_eval_preds.csv')
print(f'eval: {len(ev)} images in {(time.time() - t0) / 60:.1f} min')
tr = run(df[df.role == 'vlm_train'].sample(100, random_state=0), '/kaggle/working/vlm_train_check.csv')
print('train check:', len(tr), 'images')

## Step 7 - Quick look (full comparison is in notebook 06)

In [ ]:
from sklearn.metrics import roc_auc_score
def summary(d, name):
    y = (d.label == 'defect').astype(int)
    det = d.pred_defective.astype(str).str.lower().eq('true').astype(int)
    print(f"{name:12s} n={len(d):4d} | valid JSON {d.valid_json.mean():.1%} | "
          f"defective acc {(det == y).mean():.1%} | AUROC {roc_auc_score(y, d.score):.3f}")
summary(ev, 'EVAL')
summary(tr, 'TRAIN (100)')
print('A big gap between TRAIN and EVAL = memorisation.')